In [ ]:
# Import necessary libraries
import os
from ultralytics import YOLO
import numpy as np
import cv2
# Load the trained model
MODEL_PATH = os.path.expanduser('~/TrafficRouting/detection/models/yolo11x_visdrone_50epoch.pt')
model = YOLO(MODEL_PATH)

In [ ]:
# Traffic weights
WEIGHTS = {
    "car": 1.0,
    "van": 1.5,
    "truck": 3.0,
    "bus": 3.0,
    "motor": 0.5,
    "bicycle": 0.3
}

In [ ]:
# Function to analyze traffic in an image
def analyze_image(image_path):
    results = model.predict(source=image_path, conf=0.25, verbose=False)[0]

    traffic_counts = {k: 0 for k in WEIGHTS.keys()}
    boxes = results.boxes

    # Count objects
    for box in boxes:
        cls = int(box.cls[0].item())
        label = model.names[cls]

        if label in traffic_counts:
            traffic_counts[label] += 1

    # Compute traffic score
    traffic_score = sum(
        traffic_counts[k] * WEIGHTS[k] for k in traffic_counts
    )

    # Classify congestion
    if traffic_score < 10:
        level = "LOW"
    elif traffic_score < 25:
        level = "MEDIUM"
    else:
        level = "HIGH"

    return traffic_counts, traffic_score, level

In [ ]:
# Function to decide routing based on traffic analysis
def route_decision(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    results = model.predict(source=image_path, conf=0.25, verbose=False)[0]

    zones = {
        "left": 0,
        "center": 0,
        "right": 0
    }

    for box in results.boxes:
        x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
        cls = int(box.cls[0].item())
        label = model.names[cls]

        if label not in WEIGHTS:
            continue

        center_x = (x1 + x2) / 2

        if center_x < w / 3:
            zone = "left"
        elif center_x < 2 * w / 3:
            zone = "center"
        else:
            zone = "right"

        zones[zone] += WEIGHTS[label]

    best_route = min(zones, key=zones.get)

    return zones, best_route

In [ ]:
# Main function
if __name__ == "__main__":
    IMAGE_PATH = os.path.expanduser('~/TrafficRouting/detection/images/img0.jpg')

    counts, score, level = analyze_image(IMAGE_PATH)
    zones, route = route_decision(IMAGE_PATH)

    print("\n=== TRAFFIC ANALYSIS ===")
    print("Counts:", counts)
    print(f"Score: {score:.2f}")
    print("Congestion Level:", level)

    print("\n=== ROUTING ===")
    print("Zone Scores:", zones)
    print("Recommended Route:", route.upper())